# Clase 5 — Repositorio y gestión de prompts

Un prompt bien diseñado es un activo. Si lo usás una vez y lo perdés, la próxima vez empezás desde cero. En esta clase vamos a construir un sistema simple pero funcional para **organizar, versionar y documentar** prompts de forma que sean reutilizables por vos y por tu equipo.

## Contenido

| Sección | Tema |
|---|---|
| 1 | Configuración del entorno |
| 2 | Por qué documentar prompts |
| 3 | Estructura de metadata de un prompt |
| 4 | Construir el repositorio como DataFrame |
| 5 | Buscar, comparar y versionar prompts |
| 6 | Actividad: construir tu repositorio personal |

---
## 1. Configuración del entorno

**Si es tu primera vez en este curso:**
1. Obtené tu API key en [aistudio.google.com](https://aistudio.google.com) → **Get API key**.
2. Guardala en `.env`:
   ```bash
   echo 'GEMINI_API_KEY=TU_CLAVE_AQUI' >> .env
   ```
3. Si no querés crear el archivo, la celda te la pide de forma interactiva.

In [ ]:
import os
import getpass
import json
import pandas as pd
from datetime import date

BACKEND = "gemini"
GEMINI_MODEL = "gemini-2.5-flash-lite"

if BACKEND == "gemini":
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")
    if not GEMINI_API_KEY:
        GEMINI_API_KEY = getpass.getpass("Ingresá tu API key de Gemini: ")

print(f"Backend: {BACKEND}")

In [ ]:
if BACKEND == "gemini":
    from google import genai
    from google.genai import types
    _cliente_gemini = genai.Client(api_key=GEMINI_API_KEY)
elif BACKEND == "local":
    from huggingface_hub import hf_hub_download
    from llama_cpp import Llama
    ruta_modelo = hf_hub_download(
        repo_id="unsloth/gemma-3-1b-it-GGUF",
        filename="gemma-3-1b-it-Q4_K_M.gguf"
    )
    _llm_local = Llama(model_path=ruta_modelo, n_ctx=2048, n_gpu_layers=0, verbose=False)


def llamar_llm(prompt, system_prompt="Sos un asistente útil y conciso.", temperature=0.7, max_tokens=200):
    if BACKEND == "gemini":
        r = _cliente_gemini.models.generate_content(
            model=GEMINI_MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=system_prompt,
                temperature=temperature,
                max_output_tokens=max_tokens,
            )
        )
        return r.text.strip()
    elif BACKEND == "local":
        r = _llm_local.create_chat_completion(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": prompt}
            ],
            temperature=temperature,
            max_tokens=max_tokens
        )
        return r["choices"][0]["message"]["content"].strip()


print(llamar_llm("Respondé solo: 'Entorno listo.'", max_tokens=10))

---
## 2. Por qué documentar prompts

Cuando un equipo empieza a usar modelos de lenguaje en su trabajo, normalmente pasa esto:

1. Cada persona tiene sus propios prompts en notas sueltas o en el historial del chat.
2. Nadie sabe qué versión de un prompt funciona mejor.
3. Cuando alguien se va del equipo, se lleva el conocimiento.
4. Nadie puede mejorar sistemáticamente algo que no está registrado.

Un repositorio de prompts resuelve estos cuatro problemas. No necesita ser sofisticado: alcanza con un formato consistente y el hábito de registrar lo que funciona.

> 💡 Un prompt documentado con contexto vale mucho más que un prompt suelto, aunque el texto sea idéntico. El contexto explica *por qué* funciona.

---
## 3. Estructura de metadata de un prompt

Cada prompt en el repositorio tiene asociada una ficha con estos campos:

| Campo | Tipo | Descripción |
|---|---|---|
| `id` | string | Identificador único (ej: `feedback-v1`) |
| `tarea` | string | Categoría de la tarea (ej: `redacción`, `análisis`, `clasificación`) |
| `descripcion` | string | Qué hace el prompt en una oración |
| `version` | int | Número de versión, empieza en 1 |
| `modelo` | string | Modelo con el que fue probado |
| `temperatura` | float | Temperatura usada en las pruebas |
| `score` | int (1–5) | Calidad subjetiva de los resultados |
| `fecha` | string | Fecha de última actualización |
| `notas` | string | Qué se mejoró, qué no funciona, advertencias |

In [ ]:
# ─── Estructura de un prompt documentado ─────────────────────────────────────
# Cada entrada es un diccionario con el texto del prompt y su metadata.

def crear_entrada(id, tarea, descripcion, prompt_texto, system_prompt,
                  temperatura, score, notas, version=1, modelo=None):
    """Crea una entrada estandarizada para el repositorio."""
    return {
        "id":           id,
        "tarea":        tarea,
        "descripcion":  descripcion,
        "prompt":       prompt_texto,
        "system":       system_prompt,
        "version":      version,
        "modelo":       modelo or GEMINI_MODEL if BACKEND == "gemini" else "local",
        "temperatura":  temperatura,
        "score":        score,
        "fecha":        str(date.today()),
        "notas":        notas,
    }


print("Función crear_entrada lista.")

---
## 4. Construir el repositorio como DataFrame

Vamos a cargar el repositorio con varios prompts representativos de distintas categorías. Algunos son versiones anteriores (`v1`) de prompts que después mejoramos (`v2`), para demostrar el versionado.

In [ ]:
# ─── Prompts de ejemplo para el repositorio ───────────────────────────────────

repositorio = [
    crear_entrada(
        id="feedback-v1",
        tarea="redacción",
        descripcion="Feedback constructivo para empleado — versión inicial",
        prompt_texto="Escribí un feedback para {nombre} sobre {situacion}.",
        system_prompt="Sos un líder de equipo.",
        temperatura=0.7, score=2,
        notas="Demasiado genérico. No especifica estructura ni tono."
    ),
    crear_entrada(
        id="feedback-v2",
        tarea="redacción",
        descripcion="Feedback constructivo para empleado — con CoT y rúbrica",
        prompt_texto="""Escribí feedback para {nombre}. Situación: {situacion}
Criterios: primero lo positivo, luego el área de mejora como hecho (no juicio),
proponer una acción concreta. Tono directo. Máximo 5 oraciones.""",
        system_prompt="Sos un líder de equipo con experiencia en desarrollo de personas.",
        temperatura=0.5, score=5,
        notas="Versión estable. Probado en 12 casos reales con buen resultado.",
        version=2
    ),
    crear_entrada(
        id="sentiment-v1",
        tarea="clasificación",
        descripcion="Clasificación de sentimiento de comentarios de clientes",
        prompt_texto="""Clasificá el sentimiento: '{comentario}'
Formato: Sentimiento: <Positivo|Negativo|Neutro> | Razón: <una frase>""",
        system_prompt="Sos un analista de experiencia del cliente.",
        temperatura=0.3, score=4,
        notas="Funciona bien. Occasionally clasifica como Neutro casos borderline Negativo."
    ),
    crear_entrada(
        id="resumen-ejecutivo-v1",
        tarea="resumen",
        descripcion="Resumen ejecutivo de reportes o documentos extensos",
        prompt_texto="""Resumí el siguiente texto en formato ejecutivo:
1. Objetivo principal (1 oración)
2. Hallazgos clave (3 bullets)
3. Próximos pasos recomendados (2 bullets)

Texto: {texto}""",
        system_prompt="Sos un consultor de management que comunica con claridad.",
        temperatura=0.4, score=4,
        notas="Muy útil para reuniones. Ajustar número de bullets según longitud del texto."
    ),
    crear_entrada(
        id="sql-v1",
        tarea="código",
        descripcion="Generar consultas SQL a partir de preguntas en lenguaje natural",
        prompt_texto="""Generá una consulta SQL para esta necesidad: '{necesidad}'
Esquema disponible: {esquema}
Reglas: solo SELECT, sin subconsultas anidadas, incluir comentario de la lógica.""",
        system_prompt="Sos un DBA que escribe SQL claro y eficiente.",
        temperatura=0.2, score=4,
        notas="Temperatura baja para mayor determinismo. Validar siempre antes de ejecutar."
    ),
]

# Convertir a DataFrame para visualización y búsqueda cómodas
df = pd.DataFrame(repositorio)
print(f"Repositorio cargado: {len(df)} prompts")
df[["id", "tarea", "descripcion", "version", "score", "fecha"]]

---
## 5. Buscar, comparar y versionar prompts

In [ ]:
# ─── Buscar por tarea ─────────────────────────────────────────────────────────
def buscar_por_tarea(df, tarea):
    resultado = df[df["tarea"] == tarea][["id", "descripcion", "version", "score", "notas"]]
    print(f"Prompts para tarea '{tarea}':")
    print(resultado.to_string(index=False))


buscar_por_tarea(df, "redacción")

In [ ]:
# ─── Recuperar el mejor prompt de una tarea ───────────────────────────────────
def obtener_mejor_prompt(df, tarea):
    """Devuelve el prompt con mayor score para una tarea dada."""
    subset = df[df["tarea"] == tarea]
    if subset.empty:
        return None
    mejor = subset.loc[subset["score"].idxmax()]
    return mejor


mejor_feedback = obtener_mejor_prompt(df, "redacción")
print("Mejor prompt de redacción:")
print(f"  ID:      {mejor_feedback['id']}")
print(f"  Score:   {mejor_feedback['score']}")
print(f"  Notas:   {mejor_feedback['notas']}")
print(f"  Prompt:  {mejor_feedback['prompt'][:100]}...")

In [ ]:
# ─── Ejecutar un prompt del repositorio con variables ─────────────────────────
def ejecutar_desde_repo(df, prompt_id, variables: dict, max_tokens=200):
    """Busca el prompt por ID, rellena las variables y lo ejecuta."""
    fila = df[df["id"] == prompt_id]
    if fila.empty:
        raise ValueError(f"No existe un prompt con ID '{prompt_id}'")

    entrada = fila.iloc[0]
    texto   = entrada["prompt"].format(**variables)   # rellena las {variables}
    sistema = entrada["system"]
    temp    = entrada["temperatura"]

    print(f"Ejecutando: {prompt_id} (v{entrada['version']}, score {entrada['score']})")
    print("-" * 50)
    return llamar_llm(texto, system_prompt=sistema, temperature=temp, max_tokens=max_tokens)


resultado = ejecutar_desde_repo(
    df,
    prompt_id="feedback-v2",
    variables={
        "nombre": "Martín",
        "situacion": "llegó tarde a tres reuniones esta semana pero su trabajo fue impecable"
    }
)
print(resultado)

In [ ]:
# ─── Exportar el repositorio a JSON ───────────────────────────────────────────
# Guardamos el repositorio como archivo para compartir o versionar con git.

RUTA_REPO = "prompt_repositorio.json"

with open(RUTA_REPO, "w", encoding="utf-8") as f:
    json.dump(repositorio, f, ensure_ascii=False, indent=2)

print(f"Repositorio exportado a: {RUTA_REPO}")
print(f"Tamaño: {os.path.getsize(RUTA_REPO)} bytes")

In [ ]:
# ─── Importar y agregar una nueva versión de un prompt ────────────────────────
# Simulamos una mejora del prompt de clasificación de sentimiento.

with open(RUTA_REPO, encoding="utf-8") as f:
    repo_cargado = json.load(f)

# Agregar nueva versión mejorada
repo_cargado.append(
    crear_entrada(
        id="sentiment-v2",
        tarea="clasificación",
        descripcion="Clasificación de sentimiento — con confianza y acción recomendada",
        prompt_texto="""Clasificá el sentimiento de este comentario de cliente: '{comentario}'
Formato estricto:
Sentimiento: <Positivo|Negativo|Neutro>
Confianza: <Alta|Media|Baja>
Acción sugerida: <una oración sobre qué debería hacer el equipo de soporte>""",
        system_prompt="Sos un analista de experiencia del cliente con foco en acciones preventivas.",
        temperatura=0.3, score=5,
        notas="Agrega confianza y acción recomendada. Más útil para triaje de tickets.",
        version=2
    )
)

# Guardar versión actualizada
with open(RUTA_REPO, "w", encoding="utf-8") as f:
    json.dump(repo_cargado, f, ensure_ascii=False, indent=2)

df_actualizado = pd.DataFrame(repo_cargado)
print(f"Repositorio actualizado: {len(df_actualizado)} prompts")
df_actualizado[["id", "tarea", "version", "score"]]

---
## 6. Actividad: construir tu repositorio personal

Tomá los prompts que diseñaste en las clases anteriores (anatomía, errores y patrones, few-shot, CoT) y registralos en tu repositorio. Mínimo: 5 prompts de al menos 3 categorías distintas.

In [ ]:
# TODO: Completá tu repositorio con los prompts de las clases anteriores
# Usá la función crear_entrada() para cada uno.

mi_repositorio = [
    # crear_entrada(
    #     id="...",
    #     tarea="...",
    #     descripcion="...",
    #     prompt_texto="...",
    #     system_prompt="...",
    #     temperatura=0.7,
    #     score=...,
    #     notas="..."
    # ),
]

df_mio = pd.DataFrame(mi_repositorio) if mi_repositorio else pd.DataFrame()
print(f"Mi repositorio: {len(mi_repositorio)} prompts")
if not df_mio.empty:
    print(df_mio[["id", "tarea", "descripcion", "score"]].to_string(index=False))

In [ ]:
# TODO: Ejecutá al menos un prompt de tu repositorio con datos reales
# Usá ejecutar_desde_repo() o llamar_llm() directamente.

# Ejemplo:
# print(ejecutar_desde_repo(df_mio, "tu-prompt-id", {"variable": "valor"}))

---
## Entregable

Guardá el notebook con las celdas ejecutadas.
El entregable es el `mi_repositorio` completo (mínimo 5 prompts, 3 categorías) más al menos una ejecución real con datos.

**Este repositorio es la base del Proyecto de Módulo:** sistema de predicción simple + guía de prompts optimizados.

**Para la próxima clase:** arrancamos con Machine Learning — cuándo usarlo y cómo entrenar tu primer modelo.